In [6]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from util import summary

PATH = "../../data/ieee"  # thư mục chứa train_transaction.csv, train_identity.csv
SEED = 42
TARGET = "isFraud"
START_DATE = pd.Timestamp("2017-11-30")

PARAMS = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 256,
    "colsample_bytree": 0.7,
    "subsample": 0.7,
    "subsample_freq": 1,
    "seed": SEED,
    "n_jobs": -1,
    "verbose": -1,
}

In [2]:
df = pd.read_csv(f"{PATH}/train_transaction.csv").merge(
    pd.read_csv(f"{PATH}/train_identity.csv"), on="TransactionID", how="left"
)

f64 = df.select_dtypes("float64").columns
df[f64] = df[f64].astype("float32")

dt = START_DATE + pd.to_timedelta(df["TransactionDT"], unit="s")
df["DT_M"] = (dt.dt.year - 2017) * 12 + dt.dt.month

features = [
    c for c in df.columns if c not in ["TransactionID", "TransactionDT", TARGET, "DT_M"]
]
for c in df[features].select_dtypes(exclude="number").columns:
    df[c] = df[c].astype("category")

last = df["DT_M"].max()
train = df[df["DT_M"] < last - 1]
test = df[df["DT_M"] == last]
print("train:", train.shape, "| test:", test.shape, "| số feature:", len(features))

C:\Users\Thaibh\AppData\Local\Temp\ipykernel_1016\1040368184.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["DT_M"] = (dt.dt.year - 2017) * 12 + dt.dt.month


train: (417559, 435) | test: (89326, 435) | số feature: 431


In [3]:
dtrain = lgb.Dataset(train[features], train[TARGET])
dtest = lgb.Dataset(test[features], test[TARGET])

# lưu ý: early stopping dựa trên chính tháng test -> AUC hơi lạc quan,
# nhưng file 03 làm y hệt nên so sánh vẫn công bằng
model = lgb.train(
    PARAMS,
    dtrain,
    num_boost_round=3000,
    valid_sets=[dtest],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

pred = model.predict(test[features], num_iteration=model.best_iteration)
print("AUC:", roc_auc_score(test[TARGET], pred))

Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.905669
[200]	valid_0's auc: 0.909625
Early stopping, best iteration is:
[191]	valid_0's auc: 0.909907
AUC: 0.9099074778575292


In [4]:
imp = pd.Series(model.feature_importance("gain"), index=features).sort_values(
    ascending=False
)
imp.head(20)

V258              84968.475069
C1                40256.544400
card1             34127.148085
C13               29940.903504
card2             29873.743873
TransactionAmt    29337.951301
addr1             25751.259673
C14               25028.606811
V294              21870.124429
DeviceInfo        20030.315338
D2                15144.219828
D15               14659.355646
card5             14187.413428
R_emaildomain     13521.301390
dist1             11568.243779
D1                10858.988035
D4                10487.991393
V257               9391.999520
P_emaildomain      9192.092330
id_31              8863.059264
dtype: float64

In [ ]:
summary(test[TARGET], pred)

accuracy_score 0.9741396681817164
recall_score: 0.3333333333333333
precision_score: 0.8160377358490566
f1_score 0.47332421340629277
ROC-AUC score: 0.9099074778575292
PRAUC score: 0.540502123838962
COnfusion matrix: [[85978   234]
 [ 2076  1038]]
